# Step 11: Semantic Memory and RAG (Retrieval-Augmented Generation)

This notebook demonstrates how to store facts in memory, search them semantically, and use them to improve LLM responses using RAG.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [6]:
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import (
    AzureTextEmbedding,
    AzureChatCompletion,
)
from semantic_kernel.memory import SemanticTextMemory, VolatileMemoryStore

## Initialize Kernel and Services

We set up the Semantic Kernel with:
1. **AzureTextEmbedding**: Converts text into embeddings (vector representations)
2. **AzureChatCompletion**: Chat-based LLM for generating responses

In [7]:
kernel = Kernel()

embedding_service = AzureTextEmbedding(service_id="embedding")
chat_service = AzureChatCompletion(service_id="chat")
kernel.add_service(embedding_service)
kernel.add_service(chat_service)

## Understanding Embeddings

Let's see how embeddings work with two similar sentences.

In [8]:
TEXTS = [
    "A dog ran joyfully through the green field, chasing after butterflies in the warm afternoon sun.",
    "A happy puppy sprinted across the grassy meadow, playfully pursuing insects under the bright sky.",
]

# Generate embeddings for similar texts
text_embedded = await embedding_service.generate_embeddings(TEXTS)
print("🔢 Embeddings generated for similar texts")
print(f"Embedding dimensions: {len(text_embedded[0])}")

🔢 Embeddings generated for similar texts
Embedding dimensions: 1536


In [9]:
print(text_embedded)

[[-0.00936657 -0.00138434  0.00151592 ... -0.01851883  0.00100221
  -0.01835495]
 [-0.00157558 -0.00144804  0.00797848 ... -0.02656537  0.00446136
  -0.02239551]]


## Set Up Semantic Memory

We use `SemanticTextMemory` with `VolatileMemoryStore` (in-memory, temporary storage).

In [10]:
memory = SemanticTextMemory(
    storage=VolatileMemoryStore(), 
    embeddings_generator=embedding_service
)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23752\1488111371.py:2: DeprecationWarning: This class will be removed in a future version. Please use the InMemoryStore and Collection instead.
  storage=VolatileMemoryStore(),
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23752\1488111371.py:1: DeprecationWarning: This class will be removed in a future version.
  memory = SemanticTextMemory(


## Save Facts to Memory

Let's store some travel-related facts.

In [11]:
await memory.save_information(
    collection="travel_notes",  
    id="note1", 
    text="User is currently in Barcelona.", 
)
await memory.save_information(
    collection="travel_notes",
    id="note2",
    text="User enjoys modern art museums and seaside walks.",
)
await memory.save_information(
    collection="travel_notes",
    id="note3",
    text="Today is Saturday and the user is free in the afternoon.",
)

print("✅ Facts saved to memory!")

✅ Facts saved to memory!


In [12]:
await memory.save_information(
    collection="travel_notes",
    id="note4",
    text=TEXTS[0],
)
await memory.save_information(
    collection="travel_notes",
    id="note5",
    text=TEXTS[1],
)

print("✅ Similar texts saved to memory!")

✅ Similar texts saved to memory!


## Semantic Search

Now we'll search memory for relevant facts using semantic similarity.

In [13]:
query = "What should I recommend for this afternoon?"

results = await memory.search(collection="travel_notes", query=query, limit=3)

print(f"\n🔍 Semantic Query: {query}")
for r in results:
    print(f"✅ Match: '{r.text}' (score: {r.relevance:.2f})")


🔍 Semantic Query: What should I recommend for this afternoon?
✅ Match: 'Today is Saturday and the user is free in the afternoon.' (score: 0.79)
✅ Match: 'User enjoys modern art museums and seaside walks.' (score: 0.75)
✅ Match: 'A dog ran joyfully through the green field, chasing after butterflies in the warm afternoon sun.' (score: 0.75)


## RAG: Response WITH Memory Context

Let's use the top match as context for the LLM.

In [17]:
user_preferences="User likes outdoor activities and cultural experiences, currently the user lives in Bangalore."
context_info = results[0].text + ", " + results[1].text + ", " + results[2].text
prompt_with_context = f"User Preferences: {user_preferences}. Based on this context: '{context_info}', what can I suggest to do this afternoon?"
response_with_context = await kernel.invoke_prompt(prompt_with_context)

print("\n--- 🧠 LLM Response WITH Memory Context ---")
print(response_with_context)


--- 🧠 LLM Response WITH Memory Context ---
Given your preferences and the context, here are a few suggestions for your Saturday afternoon in Bangalore:

1. **Visit the National Gallery of Modern Art**: Spend some time exploring modern art collections that align with your interests in cultural experiences. The gallery often hosts various exhibitions that showcase contemporary and traditional art.

2. **Take a Walk at Cubbon Park**: Enjoy a leisurely stroll in one of Bangalore's largest green spaces. Although not seaside, it's perfect for enjoying the outdoors and relaxing amidst nature. You may even see some dogs frolicking through the fields, adding to the charm of your walk.

3. **Explore the HAL Heritage Centre and Aerospace Museum**: If you're interested in combining outdoor activities with a bit of cultural exploration, this museum offers a fascinating glimpse into India's aviation history. It also has outdoor exhibits, which allow you to enjoy the afternoon sun.

4. **Attend a Cu

## Response WITHOUT Memory Context

For comparison, let's ask the same question without context.

In [18]:
prompt_without_context = "What can I suggest to do this afternoon?"
response_without_context = await kernel.invoke_prompt(prompt_without_context)

print("\n--- ❓ LLM Response WITHOUT Memory ---")
print(response_without_context)


--- ❓ LLM Response WITHOUT Memory ---
That depends on your interests and where you are, but here are some general ideas for a pleasant afternoon:

1. **Outdoor Activities**: Go for a walk, run, or bike ride in a local park or nature reserve. Enjoy the fresh air and scenery.

2. **Explore Your City/Town**: Visit a local museum, art gallery, or historical site that you've never been to before.

3. **Cafe or Tea Time**: Spend some time at a cozy cafe, maybe with a good book or some friends.

4. **Cooking or Baking**: Try a new recipe or bake some treats to enjoy later.

5. **Gardening**: If you have a garden (or even just some pots), spend some time tending to your plants.

6. **Crafting**: Start a DIY project, like painting, knitting, or working on a scrapbook.

7. **Exercise**: Attend a yoga class, go for a swim, or try a workout routine at home.

8. **Visit the Library**: Spend some time browsing and borrow a book or some music.

9. **Movies or TV**: Watch a new movie or binge a serie

## Summary

You've now seen Retrieval-Augmented Generation (RAG) in action:
1. ✅ Store facts in semantic memory
2. 🔍 Search using semantic similarity
3. 🧠 Use retrieved context to improve LLM responses

This makes LLMs more reliable, accurate, and contextual!